In [1]:
import pandas as pd
import random
from pathlib import Path

SEED = 42
random.seed(SEED)

INTERMEDIATE_DIR = Path("../Datasets/intermediate")
MIN_CONCEPT_PAIR_SUPPORT = 2  # must match Component 3's threshold exactly
N_DOMAIN_VIOLATION = 300

In [2]:
def recompute_support_and_rejection():
    """Re-derive which concept edges support a PROMOTED domain claim vs a
    REJECTED one — same logic as Component 3 Section 8, but keeping the
    concept-pair linkage this time so we can pick real entities."""
    concept_rel = pd.read_csv(INTERMEDIATE_DIR / "concept_relationships.csv")
    domain_mappings = pd.read_csv(INTERMEDIATE_DIR / "domain_mappings.csv")
    concept_to_domain = dict(zip(domain_mappings["concept_name"], domain_mappings["domain_name"]))

    rows = []
    for _, row in concept_rel.iterrows():
        c_subj, p, c_obj = row["subject_concept"], row["predicate"], row["object_concept"]
        d_subj = concept_to_domain.get(c_subj)
        d_obj = concept_to_domain.get(c_obj)
        if d_subj and d_obj:
            rows.append({
                "domain_subj": d_subj, "predicate": p, "domain_obj": d_obj,
                "concept_subj": c_subj, "concept_obj": c_obj
            })
    support_df = pd.DataFrame(rows)

    support_counts = support_df.groupby(["domain_subj", "predicate", "domain_obj"])[
        ["concept_subj", "concept_obj"]
    ].apply(lambda g: g.drop_duplicates().shape[0]).reset_index(name="support_count")

    rejected_keys = set(
        support_counts[support_counts["support_count"] < MIN_CONCEPT_PAIR_SUPPORT][
            ["domain_subj", "predicate", "domain_obj"]
        ].itertuples(index=False, name=None)
    )

    # keep only the concept-pair rows whose domain triple is REJECTED
    rejected_support_df = support_df[
        support_df.apply(
            lambda r: (r["domain_subj"], r["predicate"], r["domain_obj"]) in rejected_keys, axis=1
        )
    ]
    return rejected_support_df

In [3]:
def category_D_domain_violation(n):
    entity_catalog = pd.read_csv(INTERMEDIATE_DIR / "entity_catalog.csv")
    entity_rel = pd.read_csv(INTERMEDIATE_DIR / "entity_relationships.csv")
    concept_mappings = pd.read_csv(INTERMEDIATE_DIR / "concept_mappings.csv")

    concept_to_entities = {}
    for _, row in concept_mappings.iterrows():
        concept_to_entities.setdefault(row["concept_name"], []).append(row["entity_name"])

    real_entity_edges = set(zip(entity_rel["subject"], entity_rel["predicate"], entity_rel["object"]))
    # NEW: also exclude reverse-direction matches for symmetric interacts_with
    real_entity_edges_reverse = set(
        (o, p, s) for s, p, o in real_entity_edges if p == 'interacts_with'
    )
    all_real_edges = real_entity_edges | real_entity_edges_reverse

    rejected_support_df = recompute_support_and_rejection()
    print(f"Concept-pair rows backing a REJECTED domain claim: {len(rejected_support_df)}")

    rows = rejected_support_df.sample(frac=1, random_state=SEED).to_dict("records")
    results = []

    for r in rows:
        if len(results) >= n:
            break
        c_subj, p, c_obj = r["concept_subj"], r["predicate"], r["concept_obj"]
        subj_pool = concept_to_entities.get(c_subj, [])
        obj_pool = concept_to_entities.get(c_obj, [])
        if not subj_pool or not obj_pool:
            continue

        s = random.choice(subj_pool)
        o = random.choice(obj_pool)
        if s == o or (s, p, o) in all_real_edges:   # FIXED: was only checking forward direction
            continue

        results.append({
            "subject": s, "predicate": p, "object": o,
            "ground_truth": "HALLUCINATION", "category": "domain_violation",
            "generation_rule": f"concept_edge_exists({c_subj}->{c_obj}) "
                               f"but domain_claim_rejected_K{MIN_CONCEPT_PAIR_SUPPORT}"
        })

    return results

In [4]:
def main():
    D = category_D_domain_violation(N_DOMAIN_VIOLATION)
    df = pd.DataFrame(D)
    print(f"\nGenerated {len(df)} category D facts")

    if len(df) == 0:
        print("ERROR: 0 facts generated — check that entity_catalog.csv / "
              "concept_mappings.csv have entries for the rejected concept pairs.")
        return

    # merge with the existing A/B/C set
    existing = pd.read_csv(INTERMEDIATE_DIR / "test_facts_large.csv")
    combined = pd.concat([existing, df], ignore_index=True)
    combined = combined.sample(frac=1, random_state=SEED).reset_index(drop=True)

    out_path = INTERMEDIATE_DIR / "test_facts_large_v2.csv"
    combined.to_csv(out_path, index=False)

    print(f"\nFinal composition:")
    print(combined.groupby(["category", "ground_truth"]).size())
    print(f"\nSaved {len(combined)} total facts to {out_path}")

In [5]:
main()

Concept-pair rows backing a REJECTED domain claim: 112

Generated 75 category D facts

Final composition:
category                  ground_truth 
domain_violation          HALLUCINATION     75
nonexistent_relationship  HALLUCINATION    300
real_edge                 VERIFIED         400
wrong_predicate           HALLUCINATION    200
dtype: int64

Saved 975 total facts to ../Datasets/intermediate/test_facts_large_v2.csv


In [6]:
import pandas as pd
df = pd.read_csv("../Datasets/intermediate/test_facts_large_v2.csv")
print(f"Total rows: {len(df)}")
print(f"Duplicate (subject, predicate, object) rows: {df.duplicated(subset=['subject','predicate','object']).sum()}")
print(df.groupby(["category", "ground_truth"]).size())

Total rows: 975
Duplicate (subject, predicate, object) rows: 1
category                  ground_truth 
domain_violation          HALLUCINATION     75
nonexistent_relationship  HALLUCINATION    300
real_edge                 VERIFIED         400
wrong_predicate           HALLUCINATION    200
dtype: int64


In [7]:
import pandas as pd
df = pd.read_csv("../Datasets/intermediate/test_facts_large_v2.csv")
before = len(df)
df = df.drop_duplicates(subset=['subject', 'predicate', 'object'], keep='first')
after = len(df)
print(f"Removed {before - after} duplicate rows: {before} -> {after}")
df.to_csv("../Datasets/intermediate/test_facts_large_v2.csv", index=False)
print(df.groupby(["category", "ground_truth"]).size())

Removed 1 duplicate rows: 975 -> 974
category                  ground_truth 
domain_violation          HALLUCINATION     75
nonexistent_relationship  HALLUCINATION    299
real_edge                 VERIFIED         400
wrong_predicate           HALLUCINATION    200
dtype: int64
